# ML Laboratuvarı · Modül 1, Ünite 10: Gerçek veriyi hazırlamak

**Prof. Dr. Mehmet Songur** · ML Laboratuvarı, Modül 1

Bu atölyede FRED'den aylık frekansta üç ABD serisini birleştirecek, eksik değerleri doğru ele alacak ve değişkenleri standartlaştıracaksın. Kodu **Gemini'ye yazdır**, sonra **denetle**.

| Tablo | FRED kodu | Değişken | Başlangıç |
|---|---|---|---|
| `issizlik` | UNRATE | İşsizlik oranı (%) | 1948 |
| `faiz` | FEDFUNDS | Federal fon faizi (%) | 1954 |
| `vade_farki` | T10Y3MM | 10 yıllık − 3 aylık tahvil faizi (puan) | 1982 |

**Kurallar**
1. **Kurulum** ve **Kontrol** hücrelerini değiştirme; yalnızca çalıştır.
2. Görev hücrelerinde kodu Gemini ile üret (hücredeki Gemini düğmesi ya da Gemini paneli; adı Colab'ın arayüzüne göre değişebilir).
3. Gemini'nin kodunu **çalıştırmadan önce oku**.
4. Kontrol hücresi geçerse bir **tamamlama kodu** verir; onu ML Laboratuvarı'ndaki ilgili kutuya yapıştır.

> Kendi kopyanı kaydetmek için: *Dosya → Drive'a kopya kaydet*.

In [ ]:
#@title Kurulum — çalıştır, değiştirme

import io, base64, urllib.request
import numpy as np
import pandas as pd

def fred_indir(seri_kodu):
    """FRED'den bir seriyi indirir; sütunlar: tarih, <seri_kodu>. FRED eksik değerleri '.' ile gösterir;
    bunlar bilerek NaN yapılır ve sayısı yazdırılır."""
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + seri_kodu
    with urllib.request.urlopen(url) as yanit:
        df = pd.read_csv(io.BytesIO(yanit.read()))
    df.columns = ["tarih", seri_kodu]
    df["tarih"] = pd.to_datetime(df["tarih"])
    df[seri_kodu] = pd.to_numeric(df[seri_kodu], errors="coerce")
    print(f"{seri_kodu}: {len(df)} satır, {df[seri_kodu].isna().sum()} eksik değer, {df['tarih'].min():%Y-%m} – {df['tarih'].max():%Y-%m}")
    return df

issizlik = fred_indir("UNRATE").rename(columns={"UNRATE": "issizlik"})
faiz = fred_indir("FEDFUNDS").rename(columns={"FEDFUNDS": "faiz"})
vade_farki = fred_indir("T10Y3MM").rename(columns={"T10Y3MM": "vade_farki"})
_kopya = {"issizlik": issizlik.copy(), "faiz": faiz.copy(), "vade_farki": vade_farki.copy()}
SUTUNLAR = ["issizlik", "faiz", "vade_farki"]

def _k(s):
    return base64.b64decode(s).decode()

def _beklenen():
    d = _kopya["issizlik"].merge(_kopya["faiz"], on="tarih", how="outer").merge(_kopya["vade_farki"], on="tarih", how="outer")
    return d.sort_values("tarih").reset_index(drop=True)

def kontrol_A(veri):
    b = _beklenen()
    yok = [c for c in ["tarih"] + SUTUNLAR if c not in veri.columns]
    if yok:
        print("❌ Şu sütunlar bulunamadı:", yok); return
    if len(veri) < len(b):
        print(f"❌ Satır sayısı eksik: {len(b)} olmalıydı, {len(veri)} oldu. İç birleştirme (inner) yapılmış olabilir; how='outer' kullandın mı?"); return
    if len(veri) > len(b):
        print(f"❌ Satır sayısı fazla: {len(b)} olmalıydı, {len(veri)} oldu. Tekrarlanan tarihler var mı?"); return
    if not pd.Series(veri["tarih"]).is_monotonic_increasing:
        print("❌ Tablo tarihe göre sıralı değil."); return
    v = veri.reset_index(drop=True)
    if v["tarih"].duplicated().any():
        print("❌ Bazı tarihler birden çok kez geçiyor; tarih anahtarı tekil olmalı."); return
    if not (pd.to_datetime(v["tarih"]).to_numpy() == b["tarih"].to_numpy()).all():
        print("❌ Tarih anahtarları beklenen tabloyla birebir eşleşmiyor (eksik, fazla ya da kaymış aylar)."); return
    for c in SUTUNLAR:
        if int(v[c].isna().sum()) != int(b[c].isna().sum()):
            print(f"❌ '{c}' sütunundaki eksik sayısı beklenenden farklı. Eksikleri doldurdun ya da sildin mi?"); return
        if not np.allclose(v[c].to_numpy(dtype=float), b[c].to_numpy(dtype=float), equal_nan=True):
            print(f"❌ '{c}' sütunundaki değerler beklenenden farklı."); return
    print("✅ Tüm kontroller geçti. Tamamlama kodun: " + _k("TUwxLUJSTC02UFdO"))

def kontrol_B(ortak, olcekli):
    b = _beklenen().dropna(subset=SUTUNLAR).reset_index(drop=True)
    yok = [c for c in ["tarih"] + SUTUNLAR if c not in ortak.columns]
    if yok:
        print("❌ 'ortak' tablosunda şu sütunlar yok:", yok); return
    if len(ortak) > len(b):
        print("❌ 'ortak' beklenenden uzun: eksikleri doldurmuş olabilirsin (fillna, ffill). Ortak dönem, üç serinin de gözlendiği aylardır."); return
    if len(ortak) < len(b):
        print("❌ 'ortak' beklenenden kısa: üç serinin de bulunduğu bazı aylar kaybolmuş."); return
    if ortak[SUTUNLAR].isna().any().any():
        print("❌ 'ortak' tablosunda hâlâ eksik değer var."); return
    if pd.Timestamp(ortak["tarih"].min()) != pd.Timestamp(b["tarih"].min()):
        print("❌ Ortak dönemin başlangıcı beklenenden farklı."); return
    ot = ortak.sort_values("tarih").reset_index(drop=True)
    if ot["tarih"].duplicated().any():
        print("❌ 'ortak' tablosunda bazı aylar birden çok kez geçiyor."); return
    if not (pd.to_datetime(ot["tarih"]).to_numpy() == b["tarih"].to_numpy()).all():
        print("❌ 'ortak' tablosundaki aylar, üç serinin birlikte gözlendiği aylarla birebir eşleşmiyor."); return
    if not np.allclose(ot[SUTUNLAR].to_numpy(dtype=float), b[SUTUNLAR].to_numpy(dtype=float)):
        print("❌ 'ortak' tablosundaki ham değerler kaynakla eşleşmiyor. Değerler değiştirilmiş ya da sütunlar karışmış olabilir."); return
    yok = [c for c in SUTUNLAR if c not in olcekli.columns]
    if yok:
        print("❌ 'olcekli' tablosunda şu sütunlar yok:", yok); return
    if len(olcekli) != len(ortak):
        print("❌ 'olcekli' ile 'ortak' aynı sayıda satır içermeli."); return
    if "tarih" in olcekli.columns:
        olcekli = olcekli.sort_values("tarih")
        if not (pd.to_datetime(olcekli["tarih"]).to_numpy() == b["tarih"].to_numpy()).all():
            print("❌ 'olcekli' tablosunun tarihleri 'ortak' ile hizalı değil."); return
    o = olcekli[SUTUNLAR].astype(float).reset_index(drop=True)
    if (o.mean().abs() > 1e-6).any():
        print("❌ Ölçeklenmiş sütunların ortalaması 0 değil."); return
    if ((o.std() - 1).abs() > 0.01).any():
        print("❌ Ölçeklenmiş sütunların standart sapması 1 değil. var() yerine std() mi kullanılmalıydı?"); return
    z = (b[SUTUNLAR] - b[SUTUNLAR].mean()) / b[SUTUNLAR].std()
    if not np.allclose(o.to_numpy(), z.to_numpy(), atol=0.01):
        print("❌ Ölçeklenmiş değerler beklenenden farklı. Tablo tarihe göre sıralı mı?"); return
    _aylar = pd.period_range(b["tarih"].min(), b["tarih"].max(), freq="M")
    _bos = list(_aylar.difference(pd.PeriodIndex(b["tarih"].dt.to_period("M"))).astype(str))
    print(f"Ortak gözlemler: {b['tarih'].min():%Y-%m} – {b['tarih'].max():%Y-%m}, {len(b)} ay; dönem içinde eksik aylar: {_bos or 'yok'}")
    print("✅ Tüm kontroller geçti. Tamamlama kodun: " + _k("TUwxLU9MQy0zS0pW"))

print("Tablolar hazır: issizlik, faiz, vade_farki")
issizlik.head()

## Görev A — Birleştir

Üç tabloyu `tarih` sütunu üzerinden, **hiçbir ay kaybolmayacak biçimde** birleştir, tarihe göre sırala ve sonucu `veri` adlı tabloya ata. Eksik değerleri **doldurma ya da silme**; her sütundaki eksik sayısını yazdır.

**Gemini'ye örnek istek (kendi isteğini yazman daha iyi):**
> issizlik, faiz ve vade_farki adlı üç pandas tablosu var; her birinde 'tarih' (aylık, datetime) ve bir değer sütunu bulunuyor. Üçünü 'tarih' üzerinden dış birleştirmeyle (how='outer') birleştir, tarihe göre sırala ve 'veri' adlı tabloya ata. Eksikleri doldurma veya silme. Satır sayısını ve her sütundaki eksik değer sayısını yazdır.

**Denetim soruları:** `how` ne? Satır sayısı en uzun seriden az mı? Eksik sayıları, serilerin başlangıç tarihleriyle tutarlı mı?

In [ ]:
# Görev A: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol A — çalıştır, değiştirme

In [ ]:
kontrol_A(veri)

## Görev B — Ortak dönem ve ölçek

1. `veri` tablosundan **üç serinin de bulunduğu ayları** al ve `ortak` adlı tabloya ata. Eksikleri **doldurma**: 1954 öncesi federal fon faizi yok; doldurmak, var olmayan bir geçmiş uydurmak olur.
2. `ortak` tablosundaki üç değişkeni standartlaştır (z = (x − ortalama) / standart sapma) ve `olcekli` adlı tabloya yaz. `tarih` sütunu kalabilir.

**Denetim soruları:** `fillna` ya da `ffill` var mı? Ortak dönem hangi ayda başlıyor ve hangi ayda bitiyor? Dönem içinde eksik ay var mı? Ölçeklemede `std()` mi kullanılmış? Ortalamalar 0, standart sapmalar 1 mi?

**Ortak gözlem kümesi kesintisiz bir dönem değildir.** Bir seride dönem içinde tek bir ay eksikse o ay ortak tablodan düşer. Sonraki bir gecikme hesabında satır sırası her zaman takvim ayı demek değildir. Kontrol etmek için:
```python
aylar = pd.period_range(ortak["tarih"].min(), ortak["tarih"].max(), freq="M")
print(ortak["tarih"].min(), ortak["tarih"].max(), len(ortak), list(aylar.difference(pd.PeriodIndex(ortak["tarih"].dt.to_period("M")))))
```

**Kontrol B neyi sınar?** `ortak` tablosundaki ayların üç serinin birlikte gözlendiği aylarla birebir eşleştiğini (tekrar yok), ham değerlerin kaynakla aynı olduğunu, `olcekli` tablosunun aynı aylara hizalı olduğunu (varsa `tarih` sütunu), ortalamaların 0 ve standart sapmaların 1 olduğunu (pandas `std()`, ddof=1).

In [ ]:
# Görev B: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol B — çalıştır, değiştirme

In [ ]:
kontrol_B(ortak, olcekli)

## Denetim notların

ML Laboratuvarı'ndaki **denetim defterine** geçmeden önce burada kısaca not al:
- Gemini birleştirmede hangi türü kullandı? Değiştirmen gerekti mi?
- Eksik değerlerle ne yaptın ve neden?
- Ölçeklemeyi nasıl doğruladın?